In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np
import xarray as xr
import seaborn as sns
from qgsw import plots
import matplotlib.pyplot as plt
import cycler

sns.set_theme("notebook",style="dark")
num_colors = 15
new_colors = sns.color_palette("Paired", num_colors)
plt.rcParams['axes.prop_cycle'] = cycler.cycler(color=new_colors)

In [ ]:
vorticity = np.load("../output/g5k/diagnostics/vorticity.npy",allow_pickle=True).item()
divergence = np.load("../output/g5k/diagnostics/divergence.npy",allow_pickle=True).item()
sst = np.load("../output/g5k/diagnostics/sst.npy",allow_pickle=True).item()

In [ ]:
color_maps = {
    "summer":"yellow",
    "autumn":"orange",
    "winter":"lightblue",
    "spring":"green",
}
seasons = ["summer", "autumn", "winter", "spring"]

In [ ]:
plt.rcParams["pdf.compression"] = 2
plt.rcParams["figure.titlesize"] = 23
plt.rcParams["figure.labelsize"] = 20
plt.rcParams["axes.labelsize"] = 20
plt.rcParams["lines.linewidth"] = 2
plt.rcParams["legend.fontsize"] = 20
plt.rcParams["legend.title_fontsize"] = 20
plt.rcParams["xtick.labelsize"] = 17
plt.rcParams["ytick.labelsize"] = 17

In [ ]:
vort_eq =r"$\frac{\mathrm{D}_g \zeta_g}{\mathrm{D} t}+ \frac{\mathrm{D}_g \zeta_a}{\mathrm{D} t} + {\bf{u}}_a \cdot \nabla \zeta_g + {\bf{u}}_a \cdot \nabla \zeta_a + \left(\zeta_g + \zeta_a\right) \delta + f \delta + {\bf{u}}\cdot\nabla f  + (\nabla^{\perp}w\cdot \frac{\partial {\bf{u}}}{\partial p} + w \frac{\partial \zeta}{\partial p})= \nabla^{\perp} \cdot {\bf{F}}_h$"
vort_eq_surf =r"$\dfrac{\mathrm{D}_g \zeta_g}{\mathrm{D} t} + f \delta + \dfrac{\mathrm{D}_g \zeta_a}{\mathrm{D} t} + {\bf{u}}_a \cdot \nabla \zeta_g +\zeta_g \delta + {\bf{u}}_a \cdot \nabla \zeta_a  + \zeta_a\delta  + {\bf{u}}\cdot\nabla f = \nabla^{\perp} \cdot {\bf{F}}_h$"

In [ ]:
fig,axs = plots.subplots(1,1,figsize=(20,10),dpi=1000)
plt.suptitle("Vorticity equation terms",y=1)
# plt.title(vort_eq_surf,pad=10,y=1.05,fontdict={"size":"large"})

filt_keys = {
    "time": False,
    "dt_vort_g": False,
    "dt_vort_a": False,
    "u_g_grad_vort_g": False,
    "u_g_grad_vort_a": False,
    "Dtg_vort_g":True,
    "Dtg_vort_a": True,
    "vort_div_u": False,
    "vort_g_div_u": True,
    "vort_a_div_u": True,
    "f_div_u": True,
    "u_grad_f": True,
    "u_a_grad_vort_g": True,
    "u_a_grad_vort_a": True,
}

fields = [k for k in vorticity["summer"]["july"].keys() if filt_keys[k]]

colors = {
    k : plt.rcParams['axes.prop_cycle'].by_key()['color'][i] for i,k in enumerate(fields)
}

sums = [(0,0) for _ in fields]

mids = {}

for i,season in enumerate(seasons):
    minimum=None
    maximum=None
    for j,month in enumerate((s:=vorticity[season])):
        if not s[month]:
            continue
        times = s[month]["time"][0]
        if minimum is None:
            minimum = min(times)
            maximum = max(times)
        else:
            minimum = min(min(times),minimum)
            maximum = max(max(times),maximum)
        for p,k in enumerate(fields):
            values = s[month][k][0]
            sums[p] = (sums[p][0] + values.sum(),sums[p][1] + len(values))
            if i == 0 and j == 0:
                plt.semilogy(times,values,label = s[month][k][1],color=colors[k])
            else:
                plt.semilogy(times,values,color=colors[k])
        plt.axvline(times[-1],color="grey",linestyle="dashed")
    if minimum is None and maximum is None:
        continue
    plt.axvspan(minimum,maximum,color=color_maps[season],alpha=0.1)
    mids[season] = (minimum + (maximum - minimum) / 2)
for season in seasons:
    if season not in  mids:
        continue
    plt.text(mids[season],axs[0,0].get_ylim()[1]*1.1,season,fontdict={"size":17})
means = np.array([s/n for s,n in sums])
sort_args = np.argsort(means,descending=True)

handles, labels = axs[0,0].get_legend_handles_labels()
handles = np.array(handles)[sort_args]
labels = np.array(labels)[sort_args]
labels = [l + f" ({m:.2e})" for l,m in zip(labels,means[sort_args])]


plt.legend(handles, labels,bbox_to_anchor=(1., 1.05))
plt.tight_layout()
plt.savefig("../output/images/diagnostics/budget_vorticity.pdf", bbox_inches="tight")
plt.show()

In [ ]:
div_eq =r"$\frac{\partial \delta}{\partial t} + {\bf{u}} \cdot \nabla \delta + \frac{Q^2}{2} + \frac{\delta^2}{2} - \nabla^{\perp} f \cdot {\bf{u}} - f \zeta + \Delta \Phi + (\nabla w \cdot \frac{\partial {\bf{u}}}{\partial p} + w \frac{\partial \delta}{\partial p})= \nabla_h \cdot {\bf{F}}_h$"
div_eq_surf =r"$ -f \zeta + \Delta \Phi + \dfrac{(\sigma^2 - \zeta^2)}{2} + \dfrac{\partial \delta}{\partial t} + {\bf{u}}_g \cdot \nabla \delta + {\bf{u}}_a \cdot \nabla \delta + \dfrac{\delta^2}{2} - \nabla^{\perp} f \cdot {\bf{u}} = \nabla_h \cdot {\bf{F}}_h$"

In [ ]:
fig,axs = plots.subplots(1,1,figsize=(20,10),dpi=1000)

plt.suptitle("Divergence equation terms",y=1)
# plt.title(div_eq_surf,fontdict={"size":15},pad=10, y=1.05,loc="center")

filt_keys = {
    "time":False,
    "Dt_div": False,
    "dt_div": True,
    "u_grad_div": False,
    "u_g_grad_div": True,
    "u_a_grad_div": True,
    "Q_squared": True,
    "div_squared": True,
    "sigma_squared": False,
    "zeta_squared": False,
    "f_zeta": True,
    "grad_f_u": True,
    "lap_geop": True,
    "diff_lap_geo": False,
}

fields = [k for k in divergence["summer"]["july"].keys() if filt_keys[k]]

colors = {
    k : plt.rcParams['axes.prop_cycle'].by_key()['color'][i] for i,k in enumerate(fields)
}

sums = [(0,0) for _ in fields]

mids = {}

for i,season in enumerate(seasons):
    minimum=None
    maximum=None
    for j,month in enumerate((s:=divergence[season])):
        if not s[month]:
            continue
        times = s[month]["time"][0]
        if minimum is None:
            minimum = min(times)
            maximum = max(times)
        else:
            minimum = min(min(times),minimum)
            maximum = max(max(times),maximum)
        for p,k in enumerate(fields):
            values = s[month][k][0]
            sums[p] = (sums[p][0] + values.sum(),sums[p][1] + len(values))
            if i == 0 and j == 0:
                plt.semilogy(times,values,label = s[month][k][1],color=colors[k])
            else:
                plt.semilogy(times,values,color=colors[k])
        plt.axvline(times[-1],color="grey",linestyle="dashed")
    if minimum is None and maximum is None:
        continue
    plt.axvspan(minimum,maximum,color=color_maps[season],alpha=0.1)
    mids[season] = (minimum + (maximum - minimum) / 2)
for season in seasons:
    if season not in  mids:
        continue
    plt.text(mids[season],axs[0,0].get_ylim()[1]*1.1,season,fontdict={"size":17})
means = np.array([s/n for s,n in sums])
sort_args = np.argsort(means,descending=True)

handles, labels = axs[0,0].get_legend_handles_labels()
handles = np.array(handles)[sort_args]
labels = np.array(labels)[sort_args]
labels = [l + f" ({m:.2e})" for l,m in zip(labels,means[sort_args])]


plt.legend(handles, labels,bbox_to_anchor=(1., 1))
plt.tight_layout()
plt.savefig("../output/images/diagnostics/budget_divergence.pdf", bbox_inches="tight")
plt.show()

In [ ]:
sst_eq =r"$\frac{\partial T}{\partial t} + {\bf{u}}_g \cdot \nabla T + {\bf{u}}_a \cdot \nabla T= \nabla_h \cdot {\bf{Q}}_h$"

In [ ]:
fig,axs = plots.subplots(1,1,figsize=(20,5))
plt.suptitle("SST advection equation terms",y=1.05)
plt.title(sst_eq,fontdict={"size":15},pad=18)

filt_keys = {
    "time":False,
    "dt_sst":True,
    "Dt_sst":True,
    "Dtg_sst":True,
    "u_grad_sst":True,
    "u_g_grad_sst":True,
    "u_a_grad_sst":True,
}

fields = [k for k in sst["summer"]["july"].keys() if filt_keys[k]]


colors = {
    k : plt.rcParams['axes.prop_cycle'].by_key()['color'][i] for i,k in enumerate(fields)
}

sums = [(0,0) for _ in fields]

mids = {}

for i,season in enumerate(seasons):
    minimum=None
    maximum=None
    for j,month in enumerate((s:=sst[season])):
        if not s[month]:
            continue
        times = s[month]["time"][0]
        if minimum is None:
            minimum = min(times)
            maximum = max(times)
        else:
            minimum = min(min(times),minimum)
            maximum = max(max(times),maximum)
        for p,k in enumerate(fields):
            values = s[month][k][0]
            sums[p] = (sums[p][0] + values.sum(),sums[p][1] + len(values))
            if i == 0 and j == 0:
                plt.semilogy(times,values,label = s[month][k][1],color=colors[k])
            else:
                plt.semilogy(times,values,color=colors[k])
        plt.axvline(times[-1],color="grey",linestyle="dashed")
    if minimum is None and maximum is None:
        continue
    plt.axvspan(minimum,maximum,color=color_maps[season],alpha=0.1)
    mids[season] = (minimum + (maximum - minimum) / 2)
for season in seasons:
    if season not in  mids:
        continue
    plt.text(mids[season],axs[0,0].get_ylim()[1],season)
means = np.array([s/n for s,n in sums])
sort_args = np.argsort(means,descending=True)

handles, labels = axs[0,0].get_legend_handles_labels()
handles = np.array(handles)[sort_args]
labels = np.array(labels)[sort_args]
labels = [l + f" ({m:.2e})" for l,m in zip(labels,means[sort_args])]


plt.legend(handles, labels,bbox_to_anchor=(1., 1.05))
plt.show()